In [1]:
import os
import sys
import gzip
import hashlib
import numpy as np
import pandas as pd

from tqdm import tqdm
from rdkit import Chem
from rdkit.Chem import AllChem
import lightgbm as lgb
from joblib import load

In [2]:
from rdkit.Chem.Draw import IPythonConsole
IPythonConsole.ipython_useSVG = True # Change output to SVG format
IPythonConsole.drawOptions.addAtomIndices = True
IPythonConsole.molSize = 400,400

In [3]:
from regioML.locate_EAS_sites import find_eas_sites
from HAlator.modify_smiles import remove_Hs_halator
from esnuelML.locate_atom_sites import find_nucleophilic_sites, find_electrophilic_sites

# Run predictions

In [4]:
smiles = 'CCOC(=O)c1cc(Cl)n2nc(-c3ccc(Br)cc3F)cc2n1' # from Fig.3a in https://doi.org/10.1002/anie.202411296
smiles = 'c1cnc2ccoc2c1'
smiles = Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True) # canonicalize input smiles
name = hashlib.md5(smiles.encode()).hexdigest() # SMILES MUST BE CANONICALIZED

In [8]:
# Locate sites of interest
eas_sites = find_eas_sites(Chem.MolFromSmiles(smiles))
_, pka_sites, _, _, _, _, _, _ = remove_Hs_halator(name=name, smiles=smiles, rdkit_mol=None, atomsite=None, gen_all=True, remove_H=True, rxn="rm_proton")
_, ha_sites, _, _, _, _, _, _  = remove_Hs_halator(name=name, smiles=smiles, rdkit_mol=None, atomsite=None, gen_all=True, remove_H=True, rxn="rm_hydride")
nuc_sites, nuc_names, nuc_smirks = find_nucleophilic_sites(Chem.MolFromSmiles(smiles))
elec_sites, elec_names, elec_smirks = find_electrophilic_sites(Chem.MolFromSmiles(smiles))

# Concatenate all sites
all_sites = list(set(eas_sites) | set(pka_sites) | set(ha_sites) | set(nuc_sites) | set(elec_sites))

In [91]:
sys.path.append('esnuelML')
from DescriptorCreator.GraphChargeShell import GraphChargeShell

# Calculate CM5 atomic charges
desc_generator = GraphChargeShell()
cm5_list = desc_generator.calc_CM5_charges(smiles, name=name, optimize=False, save_output=True)

# Save structures in SDF format
writer = Chem.rdmolfiles.SDWriter(desc_generator.xyz_file_path.replace('.xyz', '.sdf'))
writer.write(desc_generator.rdkit_mol)
writer.close()

all_descriptor_vectors, _ = desc_generator.create_descriptor_vector(all_sites, n_shells=3, max_neighbors=4, use_cip_sort=True)

In [92]:
pka_descriptor_vectors = [all_descriptor_vectors[all_sites.index(i)] for i in pka_sites]
ha_descriptor_vectors = [all_descriptor_vectors[all_sites.index(i)] for i in ha_sites]
nuc_descriptor_vectors = [all_descriptor_vectors[all_sites.index(i)] for i in nuc_sites]
elec_descriptor_vectors = [all_descriptor_vectors[all_sites.index(i)] for i in elec_sites]

In [105]:
pka_model = lgb.Booster(model_file='pKalculator/reg_model_all_data_dart_nshells3.txt')
ha_model = lgb.Booster(model_file='HAlator/final_reg_model_all_data_dart_default_nshells3.txt')
nuc_model = lgb.Booster(model_file='esnuelML/models/nuc/SMI2GCS_3_cm5_model.txt')
elec_model = lgb.Booster(model_file='esnuelML/models/elec/SMI2GCS_3_cm5_model.txt')

In [76]:
pka_values = pka_model.predict(pka_descriptor_vectors)
pka_values

array([39.38819966, 41.70228274, 33.59748941, 33.52948322, 37.23071006])

In [77]:
ha_values = ha_model.predict(ha_descriptor_vectors)
ha_values

array([158.05136823, 147.6913192 , 161.8237312 , 160.42565612,
       161.66236437])

In [78]:
nuc_values = nuc_model.predict(nuc_descriptor_vectors)
nuc_values

array([434.1103329 , 327.84531693, 316.54212114, 245.99446655,
       335.91802689, 349.90183857, 299.64737459, 266.78493068,
       223.13009383])

In [79]:
elec_values = elec_model.predict(elec_descriptor_vectors)
elec_values

array([122.38873511, 116.33025764,  55.04146873, 102.38945608,
        78.47480467, 198.39330078,  92.48967703, 143.15932425])

In [82]:
def find_identical_atoms(rdkit_mol, atom_list):
    len_list = len(atom_list)
    
    atom_rank = list(Chem.CanonicalRankAtoms(rdkit_mol, breakTies=False))
    for idx, atom in enumerate(rdkit_mol.GetAtoms()):
        if atom.GetIdx() in atom_list[:len_list]:
            sym_atoms = [int(atom_idx) for atom_idx, ranking in enumerate(atom_rank) if ranking == atom_rank[idx] and atom_idx not in atom_list] 
            atom_list.extend(sym_atoms)
    return atom_list


def generate_output_tables(name, rdkit_mol, sites_list, values_list, type_list, val_name='MAA Value [kJ/mol]'):
    
    sites_list_new = []
    values_list_new = []
    sdfpath_structures_new = []
    type_list_new = []
    for idx, val in enumerate(values_list):
        
        site = sites_list[idx] # the atomic index of the located site
        identical_sites = find_identical_atoms(rdkit_mol, [site])
        for site in identical_sites:
            sites_list_new.append(f'{site}')
            values_list_new.append(val)
            sdfpath_structures_new.append(f'<a href="#" class="show-structure-link" data-atom-id="{site}" sdf_path="{name}/{name}.sdf">Show</a>')
            type_list_new.append(type_list[idx].replace('_', ' ').capitalize())
    
    dict_table = {'Atom ID': sites_list_new, f'{val_name}': values_list_new, 'Reactant': sdfpath_structures_new, 'Type': type_list_new}
    df_table = pd.DataFrame(dict_table).sort_values(by=[f'{val_name}'], ascending=False)
    
    return df_table

In [ ]:
def predictor(smiles: str, name: str):

    smiles = Chem.MolToSmiles(Chem.MolFromSmiles(smiles), isomericSmiles=True) # canonicalize input smiles

    # Calculate CM5 atomic charges
    desc_generator = GraphChargeShell()
    cm5_list = desc_generator.calc_CM5_charges(smiles, name=name, optimize=False, save_output=True)

    # Save structures in SDF format
    writer = Chem.rdmolfiles.SDWriter(desc_generator.xyz_file_path.replace('.xyz', '.sdf'))
    writer.write(desc_generator.rdkit_mol)
    writer.close()

    # Locate sites of interest
    _, pka_sites, _, _, _, _, _, _ = remove_Hs_halator(name=name, smiles=smiles, rdkit_mol=None, atomsite=None, gen_all=True, remove_H=True, rxn="rm_proton")
    _, ha_sites, _, _, _, _, _, _  = remove_Hs_halator(name=name, smiles=smiles, rdkit_mol=None, atomsite=None, gen_all=True, remove_H=True, rxn="rm_hydride")
    nuc_sites, nuc_names, nuc_smirks = find_nucleophilic_sites(Chem.MolFromSmiles(smiles))
    elec_sites, elec_names, elec_smirks = find_electrophilic_sites(Chem.MolFromSmiles(smiles))

    # Concatenate all sites and generate all descriptors
    all_sites = list(set(pka_sites) | set(ha_sites) | set(nuc_sites) | set(elec_sites))
    all_descriptor_vectors, _ = desc_generator.create_descriptor_vector(all_sites, n_shells=3, max_neighbors=4, use_cip_sort=True)

    # Extract the descriptors for the different models
    pka_descriptor_vectors = [all_descriptor_vectors[all_sites.index(i)] for i in pka_sites]
    ha_descriptor_vectors = [all_descriptor_vectors[all_sites.index(i)] for i in ha_sites]
    nuc_descriptor_vectors = [all_descriptor_vectors[all_sites.index(i)] for i in nuc_sites]
    elec_descriptor_vectors = [all_descriptor_vectors[all_sites.index(i)] for i in elec_sites]

    # Run ML predictions
    pka_values = pka_model.predict(pka_descriptor_vectors)
    ha_values = ha_model.predict(ha_descriptor_vectors)
    nuc_values = nuc_model.predict(nuc_descriptor_vectors)
    elec_values = elec_model.predict(elec_descriptor_vectors)
    
    ### Generate Result Tables ###
    df_pka = generate_output_tables(name, desc_generator.rdkit_mol, list(pka_sites), pka_values, ["-" for _ in range(len(pka_values))], val_name='pKa Value').drop(columns=['Type'])
    df_ha = generate_output_tables(name, desc_generator.rdkit_mol, list(ha_sites), ha_values, ["-" for _ in range(len(ha_values))], val_name='HA Value [kcal/mol]').drop(columns=['Type'])
    df_elec = generate_output_tables(name, desc_generator.rdkit_mol, elec_sites, elec_values, elec_names, val_name='MAA Value [kJ/mol]')
    df_nuc = generate_output_tables(name, desc_generator.rdkit_mol, nuc_sites, nuc_values, nuc_names, val_name='MCA Value [kJ/mol]')
    
    # Save the tabular data to file
    df_elec.to_pickle(os.path.join(base_dir, f'data/desc_calcs/{name}/df_elec_{name}.pkl'))
    df_nuc.to_pickle(os.path.join(base_dir, f'data/desc_calcs/{name}/df_nuc_{name}.pkl'))

    return